# Create SMRI Treatment Trials Awards

Creates OpenAlex award rows from Stanley Medical Research Institute's official awarded treatment trials table and linked per-trial detail pages.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/smri_treatment_trials_to_s3.py` to fetch the official static table plus detail pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://www.stanleyresearch.org/awarded-treatment-trials/`  
**S3 location:** `s3a://openalex-ingest/awards/smri_treatment_trials/smri_treatment_trials.parquet`  
**Local full-source validation on 2026-06-08:** 416 award rows from SMRI's official Awarded Treatment Trials table; 416 unique native Grant IDs; 415/416 linked detail pages available; 100% Grant ID/title/illness/intervention/status/PI/landing-page coverage; 88.2% institution, 97.4% explicit country, 99.3% sample size, 74.5% outcome measurements, 71.9% results, 98.6% award_year parsed from Grant ID prefix. One stale detail link (`02T-152 & 02T-158`) is preserved as an index-only row.

**OpenAlex funder mapping:**
- funder_id: 4320309530
- display_name: `Stanley Medical Research Institute`
- ROR: `https://ror.org/01pj5nn22`
- DOI: `10.13039/100007123`
- Path A funder: must exist in `openalex.common.funder`.

**Mapping summary:** One OpenAlex award row per official SMRI Grant ID. `funder_award_id` is the native Grant ID (the citable award reference). `lead_investigator` is the source-published PI; `co_lead_investigator` and `investigators` are populated only when the PI field lists multiple investigators. `affiliation.country` uses only the explicit country field from the SMRI source. `amount` and `currency` are NULL because the source does not publish per-trial amounts. `start_year` is parsed from the year-coded Grant ID prefix when available; `start_date` and `end_date` remain NULL because the source does not publish day-level dates.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.smri_treatment_trials_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/smri_treatment_trials/smri_treatment_trials.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-06-08 found 416 rows.
SELECT COUNT(*) AS total_smri_trials
FROM openalex.awards.smri_treatment_trials_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.smri_treatment_trials_raw;

In [ ]:
%sql
-- Sample raw SMRI data.
SELECT
    funder_award_id,
    display_name,
    illness,
    primary_drug,
    status,
    sample_size,
    lead_name,
    institution,
    country,
    award_year,
    detail_page_found,
    landing_page_url
FROM openalex.awards.smri_treatment_trials_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. SMRI publishes no per-trial amounts; only the all-NULL
-- amount/currency columns written by the script should appear here.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'smri_treatment_trials_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant_offer|awarded|award|money|usd|eur|gbp';

In [ ]:
%sql
-- Currency-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'smri_treatment_trials_raw'
  AND LOWER(column_name) RLIKE 'currenc|currency|ccy|iso_4217|usd|eur|gbp';

In [ ]:
%sql
-- Native key uniqueness: must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.smri_treatment_trials_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id
LIMIT 20;

In [ ]:
%sql
-- Raw coverage check.
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(illness) AS has_illness,
    COUNT(primary_drug) AS has_primary_drug,
    COUNT(status) AS has_status,
    COUNT(lead_name) AS has_pi,
    COUNT(lead_family_name) AS has_pi_family,
    COUNT(institution) AS has_institution,
    COUNT(country_code) AS has_explicit_country_code,
    COUNT(sample_size) AS has_sample_size,
    COUNT(outcome_measurements) AS has_outcome_measurements,
    COUNT(results) AS has_results,
    COUNT(award_year) AS has_award_year,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(institution), COUNT(*)) * 100.0, 1) AS pct_institution,
    ROUND(try_divide(COUNT(country_code), COUNT(*)) * 100.0, 1) AS pct_explicit_country_code,
    ROUND(try_divide(COUNT(award_year), COUNT(*)) * 100.0, 1) AS pct_award_year,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year
FROM openalex.awards.smri_treatment_trials_raw;

In [ ]:
%sql
-- Program/status distribution from the source.
SELECT status, COUNT(*) AS rows
FROM openalex.awards.smri_treatment_trials_raw
GROUP BY status
ORDER BY rows DESC, status
LIMIT 25;

In [ ]:
%sql
-- Illness distribution from the source.
SELECT illness, COUNT(*) AS rows
FROM openalex.awards.smri_treatment_trials_raw
GROUP BY illness
ORDER BY rows DESC, illness
LIMIT 25;

In [ ]:
%sql
-- Country distribution. These are explicit source country values, not inferred.
SELECT country, country_code, COUNT(*) AS rows
FROM openalex.awards.smri_treatment_trials_raw
GROUP BY country, country_code
ORDER BY rows DESC, country
LIMIT 25;

In [ ]:
%sql
-- Detail-page coverage. One stale detail link is expected and preserved as an index-only row.
SELECT
    detail_page_found,
    COUNT(*) AS rows,
    ROUND(try_divide(COUNT(*), SUM(COUNT(*)) OVER ()) * 100.0, 1) AS pct_rows
FROM openalex.awards.smri_treatment_trials_raw
GROUP BY detail_page_found
ORDER BY rows DESC;

In [ ]:
%sql
-- PI frequency check for reviewer QA. Repeated PIs are plausible for this trial portfolio;
-- inspect high-frequency names with institution/title/status diversity.
SELECT
    lead_name,
    COUNT(*) AS rows,
    COUNT(DISTINCT institution) AS institutions,
    COUNT(DISTINCT display_name) AS distinct_titles,
    collect_set(status) AS statuses
FROM openalex.awards.smri_treatment_trials_raw
WHERE lead_name IS NOT NULL
GROUP BY lead_name
HAVING COUNT(*) >= 5
ORDER BY rows DESC, lead_name
LIMIT 25;

## Step 1.6: Funder Existence Assertion

In [ ]:
%sql
-- Path A funder check: this must return exactly one row.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Stanley Medical Research Institute F4320309530'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320309530;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320309530;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.smri_treatment_trials_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309530
),
raw_prepped AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        NULLIF(TRIM(display_name), '') AS display_name_norm,
        NULLIF(TRIM(description), '') AS description_norm,
        NULLIF(TRIM(status), '') AS status_norm,
        NULLIF(TRIM(lead_name), '') AS lead_name_norm,
        NULLIF(TRIM(lead_given_name), '') AS lead_given_name_norm,
        NULLIF(TRIM(lead_family_name), '') AS lead_family_name_norm,
        NULLIF(TRIM(co_lead_name), '') AS co_lead_name_norm,
        NULLIF(TRIM(co_lead_given_name), '') AS co_lead_given_name_norm,
        NULLIF(TRIM(co_lead_family_name), '') AS co_lead_family_name_norm,
        NULLIF(TRIM(institution), '') AS institution_norm,
        NULLIF(TRIM(country_code), '') AS country_code_norm,
        NULLIF(TRIM(landing_page_url), '') AS landing_page_url_norm,
        CASE
            WHEN TRY_CAST(award_year AS INT) BETWEEN 1900 AND YEAR(current_date()) + 1
                THEN TRY_CAST(award_year AS INT)
            ELSE NULL
        END AS award_year_int,
        from_json(
            other_investigators,
            'ARRAY<STRUCT<name:STRING,given_name:STRING,family_name:STRING>>'
        ) AS other_investigators_array
    FROM openalex.awards.smri_treatment_trials_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(TRY_CAST(f.funder_id AS STRING), ':', n.native_award_id))) % 9000000000 AS id,
        n.display_name_norm AS display_name,
        n.description_norm AS description,
        f.funder_id,
        TRIM(n.funder_award_id) AS funder_award_id,
        CAST(NULL AS DOUBLE) AS amount,
        CAST(NULL AS STRING) AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'research' AS funding_type,
        'Treatment Trials' AS funder_scheme,
        'smri_treatment_trials' AS provenance,
        CAST(NULL AS DATE) AS start_date,
        CAST(NULL AS DATE) AS end_date,
        CASE WHEN regexp_extract(n.funder_award_id, '^([0-9]{2})', 1) <> '' THEN (CASE WHEN CAST(regexp_extract(n.funder_award_id,'^([0-9]{2})',1) AS INT) >= 90 THEN 1900 ELSE 2000 END) + CAST(regexp_extract(n.funder_award_id,'^([0-9]{2})',1) AS INT) ELSE n.award_year_int END AS start_year,
        CAST(NULL AS INT) AS end_year,
        CASE
            WHEN n.lead_name_norm IS NOT NULL OR n.institution_norm IS NOT NULL OR n.country_code_norm IS NOT NULL THEN
                struct(
                    n.lead_given_name_norm AS given_name,
                    n.lead_family_name_norm AS family_name,
                    CAST(NULL AS STRING) AS orcid,
                    CAST(NULL AS DATE) AS role_start,
                    struct(
                        n.institution_norm AS name,
                        n.country_code_norm AS country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                    ) AS affiliation
                )
            ELSE CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
        END AS lead_investigator,
        CASE
            WHEN n.co_lead_name_norm IS NOT NULL THEN
                struct(
                    n.co_lead_given_name_norm AS given_name,
                    n.co_lead_family_name_norm AS family_name,
                    CAST(NULL AS STRING) AS orcid,
                    CAST(NULL AS DATE) AS role_start,
                    struct(
                        n.institution_norm AS name,
                        n.country_code_norm AS country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                    ) AS affiliation
                )
            ELSE CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
        END AS co_lead_investigator,
        CASE
            WHEN n.other_investigators_array IS NOT NULL THEN
                transform(
                    n.other_investigators_array,
                    x -> struct(
                        NULLIF(TRIM(x.given_name), '') AS given_name,
                        NULLIF(TRIM(x.family_name), '') AS family_name,
                        CAST(NULL AS STRING) AS orcid,
                        CAST(NULL AS DATE) AS role_start,
                        struct(
                            n.institution_norm AS name,
                            n.country_code_norm AS country,
                            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                        ) AS affiliation
                    )
                )
            ELSE CAST(NULL AS ARRAY<STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >>)
        END AS investigators,
        n.landing_page_url_norm AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT(
            'https://api.openalex.org/works?filter=awards.id:G',
            TRY_CAST(abs(xxhash64(CONCAT(TRY_CAST(f.funder_id AS STRING), ':', n.native_award_id))) % 9000000000 AS STRING)
        ) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM raw_prepped n
    CROSS JOIN funder_resolved f
)
SELECT *
FROM awards_transformed;

## Step 3: Delete and Insert at Priority 214

In [ ]:
%sql
-- Remove previous SMRI treatment-trial data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'smri_treatment_trials' AND priority = 214;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    214 AS priority
FROM openalex.awards.smri_treatment_trials_awards;

## Step 6: Verification

In [ ]:
%sql
-- Row count should match the 416-row local validation.
SELECT COUNT(*) AS total_awards
FROM openalex.awards.smri_treatment_trials_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.smri_treatment_trials_awards;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(start_year) AS has_start_year,
    COUNT(lead_investigator) AS has_lead_investigator,
    COUNT(lead_investigator.family_name) AS has_pi_family,
    COUNT(lead_investigator.affiliation.name) AS has_institution,
    COUNT(lead_investigator.affiliation.country) AS has_explicit_country,
    COUNT(landing_page_url) AS has_url,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_pi_family,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_institution,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.country), COUNT(*)) * 100.0, 1) AS pct_explicit_country
FROM openalex.awards.smri_treatment_trials_awards;

In [ ]:
%sql
-- Amount/currency waiver check. SMRI does not publish per-trial amounts; NULL is expected.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.smri_treatment_trials_awards;

In [ ]:
%sql
-- Funder consistency.
SELECT funder.display_name, COUNT(*) AS rows
FROM openalex.awards.smri_treatment_trials_awards
GROUP BY funder.display_name;

In [ ]:
%sql
-- Year distribution and future-year cap.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.smri_treatment_trials_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 30;

In [ ]:
%sql
-- Must be 0.
SELECT COUNT(*) AS future_year_rows
FROM openalex.awards.smri_treatment_trials_awards
WHERE start_year > YEAR(current_date()) + 1;

In [ ]:
%sql
-- Sample transformed awards for reviewer inspection.
SELECT
    id,
    display_name,
    funder_award_id,
    funding_type,
    funder_scheme,
    amount,
    currency,
    start_year,
    lead_investigator.given_name AS pi_given,
    lead_investigator.family_name AS pi_family,
    lead_investigator.affiliation.name AS institution,
    lead_investigator.affiliation.country AS country_code,
    landing_page_url
FROM openalex.awards.smri_treatment_trials_awards
ORDER BY start_year DESC NULLS LAST, funder_award_id
LIMIT 20;

In [ ]:
%sql
-- Verify the rows inserted at priority 214.
SELECT
    provenance,
    priority,
    COUNT(*) AS rows,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'smri_treatment_trials'
  AND priority = 214
GROUP BY provenance, priority;

## Contractor Handoff Notes

Contractor validation completed locally on 2026-06-08:
- `python3 -m py_compile scripts/local/smri_treatment_trials_to_s3.py`
- `python3 scripts/local/smri_treatment_trials_to_s3.py --limit 10 --skip-upload --output-dir /tmp/smri_smoke`
- `python3 scripts/local/smri_treatment_trials_to_s3.py --skip-upload --output-dir /tmp/smri_full`
- notebook JSON valid and outputs cleared before commit
- provenance preflight `smri_treatment_trials` returned `meta.count = 0`

Admin TODO:
- Upload the parquet by running `scripts/local/smri_treatment_trials_to_s3.py` without `--skip-upload`.
- Run this notebook on Databricks and review the Step 6 cells.
- Run `CreateAwards.ipynb` / downstream QA and mark the tracker Complete after production verification.

Contractor has no S3/Databricks access; admin must run upload + Databricks notebook + QA.
